# TP1 · From raw data to comparison vectors

*Data Matching course · M2 MIAGE · Session 1 of 3 · Student version*

| Session | Notebook | Topic |
|---|---|---|
| **1** | **TP1** | **Know the data, standardise it, build and compare candidate pairs** |
| 2 | TP2 | Decide and evaluate; discover language models |
| 3 | TP3 | Language models in the matching pipeline |

This session alternates short lectures and practice. Each part of this notebook follows one block of the lecture:

| Part | Lecture block | Practice |
|---|---|---|
| 1 | Profiling | discover the real content of *Febrl2* (≈ 15 min) |
| 2 | Standardisation | put all values in one format (≈ 20 min) |
| 3 | Pairing | choose the pairs to compare, measure the choice (≈ 30 min) |
| 4 | Comparison | one similarity score per attribute (≈ 25 min) |

Each part ends with one short exercise: **observe**, then **explain** in two or three sentences.

## The dataset

In this course, we use the *Febrl2* dataset. It is built into the *Python Record Linkage Toolkit* (https://recordlinkage.readthedocs.io/en/latest/about.html). This dataset was generated with *Febrl* (*Freely Extensible Biomedical Record Linkage*). It contains 5000 records: 4000 original records and 1000 duplicate records. Among the 4000 original records:
* 114 have 1 duplicate record,
* 141 have 2 duplicate records,
* 107 have 3 duplicate records,
* 47 have 4 duplicate records,
* 19 have 5 duplicate records,
* the other 3572 have no duplicate record.

The number of duplicates per original record follows a Poisson distribution, as shown in the figure below.

<img src="img/image1.png" width="50%"/>

Note: in the figure, the ***duplicate/original ratio*** is the number of duplicate records per original record (it is not a global ratio). For instance, ***ratio = 5*** corresponds to the 19 original records that each have ***5*** duplicate records.

A duplicate is a **modified copy** of an original record: typos, missing values, abbreviations. Our goal in the whole course is to find these duplicates again.

## 0. Set-up

In [21]:
#!pip install recordlinkage nltk
import time
import numpy as np
import pandas as pd
import recordlinkage
from recordlinkage.datasets import load_febrl2

df = load_febrl2()
raw = df.copy()      # we keep a copy of the raw data, to compare before and after standardisation
df

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-2778-org,sarah,bruhn,44,forbes street,wintersloe,kellerberrin,4510,vic,19300213,7535316
rec-712-dup-0,jacob,lanyon,5,milne cove,wellwod,beaconsfield upper,2602,vic,19080712,9497788
rec-1321-org,brinley,efthimiou,35,sturdee crescent,tremearne,scarborough,5211,qld,19940319,6814956
rec-3004-org,aleisha,hobson,54,oliver street,inglewood,toowoomba,3175,qld,19290427,5967384
rec-1384-org,ethan,gazzola,49,sheaffe street,bimby vale,port pirie,3088,sa,19631225,3832742
...,...,...,...,...,...,...,...,...,...,...
rec-1487-org,thomas,green,44,tuthill place,holmeleigh,bonny hills,4740,vic,19420210,9334580
rec-1856-org,james,mcneill,42,archibald street,NaN,evans head,2250,nsw,19011207,4837378
rec-3307-org,paige,lock,7,a'beckett street,camboon,carina heights,2290,nsw,19871002,5142242


## 1. Profiling: first glance *(lecture block "Profiling")*

There is no metadata associated with this dataset. Therefore, we carry out some descriptive analyses with basic built-in DataFrame methods. Three questions guide us:
* **Completeness.** Are values missing?
* **Uniqueness.** How many different values does each attribute have?
* **Validity.** Are the values allowed (for instance, a real Australian state)?

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, rec-2778-org to rec-1143-org
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   given_name     4891 non-null   object
 1   surname        4936 non-null   object
 2   street_number  4777 non-null   object
 3   address_1      4891 non-null   object
 4   address_2      4431 non-null   object
 5   suburb         4950 non-null   object
 6   postcode       5000 non-null   object
 7   state          4952 non-null   object
 8   date_of_birth  4890 non-null   object
 9   soc_sec_id     5000 non-null   object
dtypes: object(10)
memory usage: 429.7+ KB


The column *Non-Null Count* gives the number of non-missing values in each column. It is a count, not a constraint. Some columns have fewer than 5000 non-null values: they contain missing values (NaN), and their *completeness* is below 100%.

In [23]:
df.describe(include='all')

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
count,4891,4936,4777,4891,4431,4950,5000,4952,4890,5000
unique,907,1816,397,2455,2517,1684,1412,19,3785,4089
top,emiily,white,1,forbes street,rosetta village,toowoomba,4701,nsw,19690905,1796925
freq,85,131,142,16,40,46,23,1604,7,6


We have 4089 unique values of ***soc_sec_id***. This number is neither the total number of records (***5000***) nor the number of original records (***4000***). In fact, a duplicate record may contain a modified (for instance, mistyped) *soc_sec_id*. Therefore, ***soc_sec_id*** cannot be used as a unique and reliable identifier: we will have to compare **several** attributes.

In [24]:
# Validity: the values of "state". Australia has 8 states and territories.
df["state"].value_counts(dropna=False)

state
nsw    1604
vic    1324
qld     900
wa      476
sa      388
tas     138
act      76
NaN      48
nt       34
qdl       2
vid       1
nws       1
vkc       1
qod       1
sk        1
wz        1
viy       1
aw        1
vif       1
nss       1
Name: count, dtype: int64

The list contains more than 8 values: some of them are typos of a valid state. This is a **validity** problem.

**What does a duplicate look like?** The identifier of each record tells us which person it describes: *rec-123-org* is an original record, and *rec-123-dup-0* is a duplicate of the same person. We use this information only to **understand** the data and, later, to **evaluate** the matching. We never use it to find the duplicates.

In [25]:
person = df.index.str.extract(r"rec-(\d+)-", expand=False)
biggest = person.value_counts().index[0]     # the person with the largest number of records
df[person == biggest]

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-1213-dup-0,amber,white,1,shackleton circuit,NaN,hackham,6770,sa,19690905,7925975
rec-1213-dup-3,amber,brangwen,1,shackletoncircuit,everhreen,hackham,7607,sa,19690905,7925975
rec-1213-dup-2,amber,brangwen,1,shackletok circuit,evergeen,hackhsm,6707,sa,19690905,7925975
rec-1213-org,amber,brangwen,1,shackleton circuit,evergreen,hackham,6707,sa,19690905,7925975
rec-1213-dup-1,amber,brangwen,1,evergreen,shackleton circuit,hackham,6707,sa,19690905,7925975
rec-1213-dup-4,amger,brangwen,1,shackleton circuit,NaN,hackham,6707,sa,19690905,7925075


Compare these records attribute by attribute: typos, missing values, modified values. A simple equality test would miss most of these duplicates.

### Exercise 1 · Which attributes can help?

<font color='red'>

***To do***: build a table with, for each attribute of the raw data (`raw`), the number of missing values and the number of distinct values.
* Which attribute has the most missing values? Which one has the most distinct values?
* Which attributes look useful to find duplicates? Why?

</font>

In [26]:
profile = pd.DataFrame({
    "missing": raw.isna().sum(),
    "distinct": raw.nunique(dropna=True)
})

profile = profile.sort_values(["missing", "distinct"], ascending=False)
display(profile)

print(f"Most missing values: {profile['missing'].idxmax()} ({profile['missing'].max()})")
print(f"Most distinct values: {profile['distinct'].idxmax()} ({profile['distinct'].max()})")

,missing,distinct
address_2,569,2517
street_number,223,397
date_of_birth,110,3785
address_1,109,2455
given_name,109,907
surname,64,1816
suburb,50,1684
state,48,19
soc_sec_id,0,4089
postcode,0,1412


Most missing values: address_2 (569)
Most distinct values: soc_sec_id (4089)


*Your answer:*

...

## 2. Standardisation *(lecture block "Standardisation")*

https://recordlinkage.readthedocs.io/en/latest/ref-preprocessing.html

When all values follow the same format, we have a better chance of finding the duplicates.

#### 2.1 Uppercase

We convert all text values to uppercase. (The *Febrl2* values are already in lowercase, so this step is only an illustration.)

In [27]:
# We do NOT use df.astype(str) here. With pandas 1.x and 2.x (before pandas 3.0), astype(str)
# turns missing values (NaN) into the string "nan" (then "NAN"). Two missing values would then
# look identical, and the comparison step would give them a similarity score of 1.
df = df.apply(lambda col: col.str.upper())
df.head()

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-2778-org,SARAH,BRUHN,44,FORBES STREET,WINTERSLOE,KELLERBERRIN,4510,VIC,19300213,7535316
rec-712-dup-0,JACOB,LANYON,5,MILNE COVE,WELLWOD,BEACONSFIELD UPPER,2602,VIC,19080712,9497788
rec-1321-org,BRINLEY,EFTHIMIOU,35,STURDEE CRESCENT,TREMEARNE,SCARBOROUGH,5211,QLD,19940319,6814956
rec-3004-org,ALEISHA,HOBSON,54,OLIVER STREET,INGLEWOOD,TOOWOOMBA,3175,QLD,19290427,5967384
rec-1384-org,ETHAN,GAZZOLA,49,SHEAFFE STREET,BIMBY VALE,PORT PIRIE,3088,SA,19631225,3832742


#### 2.2 Stop words removal

Stop words are frequent words that carry little information: “the”, “a” in a sentence; “Mr”, “Mrs” in names; “Street”, “St”, “Road” in addresses. In the example below, we remove some street types from *address_1*.

In [ ]:
import nltk
# Run once to download the tokenizer models (NLTK 3.8.2 or later: 'punkt_tab'; older versions: 'punkt').
#nltk.download('punkt_tab')
from nltk.tokenize.treebank import TreebankWordDetokenizer

address_stopwords = ["STREET", "ST", "PLACE", "RD", "ROAD"]
detokenizer = TreebankWordDetokenizer()

# Missing values (NaN) are not strings: we keep them unchanged (... else x).
df['address_1_token'] = df['address_1'].apply(lambda x: nltk.word_tokenize(x) if isinstance(x, str) else x)
df['address_1_clean'] = df['address_1_token'].apply(lambda x: [w for w in x if w not in address_stopwords] if isinstance(x, list) else x)
df['address_1_clean'] = df['address_1_clean'].apply(lambda x: detokenizer.detokenize(x) if isinstance(x, list) else x)
df['address_1_clean'] = df['address_1_clean'].replace('', np.nan)   # an empty address is a missing value
df[['address_1', 'address_1_clean']].head(10)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\czy31\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,address_1,address_1_clean
rec_id,,
rec-2778-org,FORBES STREET,FORBES
rec-712-dup-0,MILNE COVE,MILNE COVE
rec-1321-org,STURDEE CRESCENT,STURDEE CRESCENT
rec-3004-org,OLIVER STREET,OLIVER
rec-1384-org,SHEAFFE STREET,SHEAFFE
rec-3981-org,MANSFIELD PLACE,MANSFIELD
rec-916-org,KEENAN STREET,KEENAN
rec-1684-org,BASEBY PLACE,BASEBY
rec-63-dup-0,DUFFY STREET,DUFFY


/!\ We must examine the data before choosing the stop words. In France, *1 rue Jean Jaurès* and *1 place Jean Jaurès* are two different places: removing the street type would create false matches.

#### 2.3 Postcodes: keep only the digits

In [29]:
df['postcode'] = df['postcode'].str.strip()
df['postcode'] = df['postcode'].str.findall(r'[0-9]+').str.join("")   # r'[0-9]+': one or several digits
df['postcode'] = df['postcode'].replace("", np.nan)                   # an empty postcode is a missing value

#### 2.4 Symbols

In [30]:
# Since pandas 2.0, str.replace() treats the pattern as plain text by default: we must write regex=True.
# Otherwise, nothing is replaced and no error is raised.
for col in ['address_1_clean', 'address_2']:
    df[col] = df[col].str.replace(r"[^\w\s]", " ", regex=True)            # symbol -> blank space
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()    # extra blank spaces
    df[col] = df[col].replace("", np.nan)                                 # empty string -> NaN
df[['given_name', 'surname', 'address_1_clean', 'address_2', 'postcode', 'state']].head(10)

,given_name,surname,address_1_clean,address_2,postcode,state
rec_id,,,,,,
rec-2778-org,SARAH,BRUHN,FORBES,WINTERSLOE,4510,VIC
rec-712-dup-0,JACOB,LANYON,MILNE COVE,WELLWOD,2602,VIC
rec-1321-org,BRINLEY,EFTHIMIOU,STURDEE CRESCENT,TREMEARNE,5211,QLD
rec-3004-org,ALEISHA,HOBSON,OLIVER,INGLEWOOD,3175,QLD
rec-1384-org,ETHAN,GAZZOLA,SHEAFFE,BIMBY VALE,3088,SA
rec-3981-org,ALICIA,HOPE,MANSFIELD,SUNSET,6061,SA
rec-916-org,BENJAMIN,KOLOSCHE,KEENAN,WINGARA,3212,SA
rec-1684-org,PETREECE,LUSTED,BASEBY,NaN,6214,SA
rec-63-dup-0,OLIVIA,WHITE,DUFFY,SHOPPING VILLAGE,2260,VIC


#### 2.5 The missing value trap

A missing value must **stay missing** (NaN). If we replace it by an empty string, two missing values become two equal strings, and the comparison step counts an agreement that proves nothing.

In [31]:
print("Two NaN values are equal?    ", np.nan == np.nan)   # False
print("Two empty strings are equal? ", "" == "")           # True: a false agreement
n = df["address_2"].isna().sum()
print(f"{n} missing address_2 -> {n * (n - 1) // 2:,} pairs would 'agree' if we wrote '' instead of NaN")

Two NaN values are equal?     False
Two empty strings are equal?  True
569 missing address_2 -> 161,596 pairs would 'agree' if we wrote '' instead of NaN


### Exercise 2 · Stop words: a careful choice

<font color='red'>

***To do***: count the distinct values of *address_1* (raw) and of *address_1_clean*. Then display the 15 most frequent **last words** of *address_1_clean*.
* Why does the number of distinct values change?
* Which street types are still there? Should we remove them too? Would you do the same with French addresses?

</font>

In [43]:
print(f"Distinct raw address_1 values: {df['address_1'].nunique(dropna=True)}")
print(f"Distinct clean address_1 values: {df['address_1_clean'].nunique(dropna=True)}")

last_words = df['address_1_clean'].dropna().str.split().str[-1]
print("\n15 most frequent last words:")
display(last_words.value_counts().head(15).to_frame("count"))

Distinct raw address_1 values: 2455
Distinct clean address_1 values: 2453

15 most frequent last words:


,count
address_1_clean,
CRESCENT,819
CIRCUIT,414
AVENUE,191
CLOSE,166
DRIVE,112
COURT,78
TREET,21
GARDEN,17
FORBES,16


*Your answer:*

...

## 3. Pairing *(lecture block "Couplage")*

https://recordlinkage.readthedocs.io/en/latest/ref-index.html

We cannot compare all pairs of records. Pairing (also called *indexing*) chooses the **candidate pairs** that we will compare.

To **measure** a pairing, we need the true duplicate pairs (the *ground truth*). We load them now, and we define a small tool. You can run the next cell without reading it in detail.

In [33]:
_, true_links = load_febrl2(return_links=True)     # the 1934 true pairs (used only to measure)
N = len(df)

def pair_keys(pairs):
    # One number per pair: the same number for (a, b) and (b, a).
    a = df.index.get_indexer(pairs.get_level_values(0))
    b = df.index.get_indexer(pairs.get_level_values(1))
    return np.minimum(a, b).astype(np.int64) * N + np.maximum(a, b)

TRUE_ALL = np.unique(pair_keys(true_links))
idx_rows = []

def indexing_quality(pairs, name):
    # Pair completeness: share of the true pairs that are kept. Reduction ratio: share of all pairs removed.
    keys = np.unique(pair_keys(pairs))
    row = {"pairing": name, "pairs": len(keys),
           "pair completeness": round(float(np.isin(TRUE_ALL, keys).mean()), 4),
           "reduction ratio": round(1 - len(keys) / (N * (N - 1) / 2), 4)}
    idx_rows.append(row)
    return row

print(f"{len(TRUE_ALL)} true pairs")

1934 true pairs


There are 1934 true pairs, not 1000: in a group of *k*+1 records of the same person, **every** pair is a true pair (a group of 3 records gives 3 pairs).

### 3.1 Full Index

A *Full Index* contains all the possible pairs. With 5000 records, it creates 5000 × 4999 / 2 = 12,497,500 pairs. We do not build them (it would use a lot of memory): we only count them.

In [34]:
print(f"{N} records -> {N * (N - 1) // 2:,} possible pairs")
print(f"At 10,000 comparisons per second: {N * (N - 1) / 2 / 10_000 / 60:.0f} minutes")

5000 records -> 12,497,500 possible pairs
At 10,000 comparisons per second: 21 minutes


<a id='blocking'></a>
### 3.2 Blocking

Blocking pairs up only the records that have exactly the same value for a *record key* (for instance, the state).

In [35]:
from recordlinkage.index import Block, SortedNeighbourhood
Block_Index_by_State_Pairs = Block(left_on="state").index(df)
print(f"Blocking on state: {len(Block_Index_by_State_Pairs):,} pairs")

Blocking on state: 2,766,975 pairs


A state contains many persons, so we still get millions of pairs. And if two records of the same person have a typo in the state, they are never compared.

### 3.3 Sorted Neighbourhood

*Sorted Neighbourhood* pairs up records with ***nearby*** values of a key: the values are sorted, and a window slides over the sorted list. With `window=5`, each distinct surname is paired with the 2 distinct surnames just before it and the 2 just after it.

In [36]:
Neighbour_Index_by_Name_Pairs = SortedNeighbourhood(left_on="surname", window=5).index(df)
print(f"Sorted Neighbourhood on surname: {len(Neighbour_Index_by_Name_Pairs):,} pairs")

Sorted Neighbourhood on surname: 72,701 pairs


The key can also be built from several attributes:
<a id='key_neighbour'></a>
<img src="img/image2.png" width="80%"/>

### 3.4 Hybrid approach, and how to measure a pairing

We combine *Blocking* and *Sorted Neighbourhood* (union). Then we answer two simple questions:
1. **Did we keep the true pairs?** (*pair completeness*: a lost true pair is lost for ever.)
2. **Did we remove enough pairs?** (*reduction ratio*.)

In [37]:
All_Index_Pairs = Block_Index_by_State_Pairs.union(Neighbour_Index_by_Name_Pairs)
indexing_quality(Block_Index_by_State_Pairs, "block(state)")
indexing_quality(Neighbour_Index_by_Name_Pairs, "SN(surname)")
indexing_quality(All_Index_Pairs, "hybrid: block(state) + SN(surname)")
del Block_Index_by_State_Pairs, All_Index_Pairs     # free memory (millions of pairs)
pd.DataFrame(idx_rows)

,pairing,pairs,pair completeness,reduction ratio
0,block(state),2766975,0.9374,0.7786
1,SN(surname),72701,0.7544,0.9942
2,hybrid: block(state) + SN(surname),2822968,0.9819,0.7741


### 3.5 Several selective keys: multi-pass pairing

A good key is **selective** (many different values), but it can contain a typo. We therefore use **several** selective keys and keep the union. A true pair is kept if its two records agree on **at least one** key.

In [38]:
passes = {
    "SN(surname)":          SortedNeighbourhood(left_on="surname", window=5),
    "SN(given_name)":       SortedNeighbourhood(left_on="given_name", window=5),
    "block(postcode)":      Block(left_on="postcode"),
    "block(date_of_birth)": Block(left_on="date_of_birth"),
    "block(soc_sec_id)":    Block(left_on="soc_sec_id"),
}
candidates = None
for name, algorithm in passes.items():
    p = algorithm.index(df)
    candidates = p if candidates is None else candidates.union(p)
    print(f"{name:22s} {len(p):>9,d} pairs | union so far: {len(candidates):>9,d}")
indexing_quality(candidates, "multi-pass (5 keys)")
pd.DataFrame(idx_rows)

SN(surname)               72,701 pairs | union so far:    72,701
SN(given_name)            99,828 pairs | union so far:   170,816
block(postcode)           14,052 pairs | union so far:   183,428
block(date_of_birth)       1,979 pairs | union so far:   183,772
block(soc_sec_id)          1,686 pairs | union so far:   183,773


,pairing,pairs,pair completeness,reduction ratio
0,block(state),2766975,0.9374,0.7786
1,SN(surname),72701,0.7544,0.9942
2,hybrid: block(state) + SN(surname),2822968,0.9819,0.7741
3,multi-pass (5 keys),183773,1.0000,0.9853


### Exercise 3 · Choose a blocking key

<font color='red'>

***To do***: measure a blocking on *postcode*, on *date_of_birth* and on *given_name* with `indexing_quality`.
* Which single key is the best compromise? Why is *state* a bad key?
* Why does the multi-pass pairing keep more true pairs than any single key?

</font>

In [ ]:
single_keys = ["postcode", "date_of_birth", "given_name"]

start = len(idx_rows)
for key in single_keys:
    pairs = Block(left_on=key).index(df)
    indexing_quality(pairs, f"block({key})")

pd.DataFrame(idx_rows[start:])

,pairing,pairs,pair completeness,reduction ratio
0,block(postcode),14052,0.7616,0.9989
1,block(date_of_birth),1979,0.8630,0.9998
2,block(given_name),48883,0.6101,0.9961


*Your answer:*

...

## 4. Comparison *(lecture block "Comparaison")*

https://recordlinkage.readthedocs.io/en/latest/ref-compare.html

For each candidate pair, we compute **one similarity score per attribute**, between 0 (different) and 1 (identical):
* **Jaro–Winkler** for names and words: tolerant to typos, bonus when the beginning is identical;
* **Levenshtein** (edit distance) for codes and numbers: counts the characters to change;
* **exact** equality for short codes (the state).

<a id='compare'></a>
We keep the scores **continuous** (no threshold): the decision step (session 2) will use them. A missing value gives 0.

In [40]:
cmp = recordlinkage.Compare()
cmp.string("given_name", "given_name", method="jarowinkler", label="given_name")
cmp.string("surname", "surname", method="jarowinkler", label="surname")
cmp.string("street_number", "street_number", method="levenshtein", label="street_number")
cmp.string("address_1_clean", "address_1_clean", method="jarowinkler", label="address_1")
cmp.string("address_2", "address_2", method="jarowinkler", label="address_2")
cmp.string("suburb", "suburb", method="jarowinkler", label="suburb")
cmp.string("postcode", "postcode", method="levenshtein", label="postcode")
cmp.exact("state", "state", label="state")
cmp.string("date_of_birth", "date_of_birth", method="levenshtein", label="date_of_birth")
cmp.string("soc_sec_id", "soc_sec_id", method="levenshtein", label="soc_sec_id")

t0 = time.time()
features = cmp.compute(candidates, df)
y = np.isin(pair_keys(features.index), TRUE_ALL)     # True for the true pairs (used only to measure)
print(f"{len(features):,} pairs x {features.shape[1]} scores in {time.time() - t0:.0f} s; {y.sum()} true pairs")
features.head()

183,773 pairs x 10 scores in 3 s; 1934 true pairs


given_name   surname  street_number  address_1  \
rec_id_1  rec_id_2                                                       
rec-0-org rec-1118-org    0.955556  0.000000       0.333333   0.511111   
          rec-1524-org    0.502646  0.422619       0.333333   0.577778   
          rec-2170-org    1.000000  0.550000       0.000000   0.455556   
          rec-2410-org    0.955556  0.430556       0.000000   0.511111   
          rec-248-org     1.000000  0.383333       0.000000   0.455556   

                        address_2    suburb  postcode  state  date_of_birth  \
rec_id_1  rec_id_2                                                            
rec-0-org rec-1118-org   0.451389  0.459707      0.25      0          0.250   
          rec-1524-org   0.523810  0.573138      1.00      0          0.250   
          rec-2170-org   0.625000  0.521368      0.25      0          0.250   
          rec-2410-org   0.427579  0.382479      0.00      0          0.250   
          rec-248-org    0.564815  0.442002      0.00      0          0.375   

                        soc_sec_id  
rec_id_1  rec_id_2                  
rec-0-org rec-1118-org    0.285714  
          rec-1524-org    0.142857  
          rec-2170-org    0.142857  
          rec-2410-org    0.142857  
          rec-248-org     0.000000

This table is the **comparison vector** of each pair: one row per pair, one score per attribute. Nothing is decided yet.

Which attributes separate the true pairs from the other pairs? We compare the **average score** of each attribute.

In [41]:
avg = features.groupby(y).mean().T
avg.columns = ["other pairs", "true pairs"]
avg["difference"] = avg["true pairs"] - avg["other pairs"]
avg.sort_values("difference", ascending=False).round(2)

,other pairs,true pairs,difference
soc_sec_id,0.13,0.94,0.81
state,0.22,0.94,0.72
postcode,0.22,0.89,0.67
street_number,0.12,0.78,0.66
date_of_birth,0.41,0.91,0.50
suburb,0.45,0.93,0.47
address_1,0.44,0.89,0.45
address_2,0.37,0.72,0.35
surname,0.60,0.85,0.26
given_name,0.65,0.84,0.19


An attribute is useful when its score is **high for the true pairs and low for the other pairs**. Two persons often live in the same state: the state score is high for many non-matches, so it is a weak clue.

### Exercise 4 · Which measure for which attribute?

<font color='red'>

***To do***: compare the surname with **Levenshtein** instead of Jaro–Winkler, on the same candidate pairs. Compute the average score for the true pairs and for the other pairs, for both measures. Which measure separates the two groups better? Why?

</font>

In [42]:
# Your code here

*Your answer:*

...

## What's next?

In session 2, we use these comparison vectors to **decide** (match or non-match), we **measure** the quality of the decisions, and we let the data **learn** the weights of the attributes. TP2 rebuilds everything from the start: you can close this notebook.